## Plotly Volume plots

by: Brett Mattas
<br>date: 5/21/2016

### Purpose

The purpose of this notebook is to demonstrate the use of plotly volume plots.

### Included:

1. Basic volume plot
2. Item
3. Item

In [1]:
import plotly.graph_objects as go
import numpy as np
from buried_structures import Point_3d, Origin
from math import pi
print("Finished Imports")

Finished Imports


In [2]:
def bous(p: Point_3d) -> float:
    # Simplified Boussinesq problem with hard coded
    # inputs that only returns downward pressure.
    # Used for demonstrating plotly charts work as
    # intended.

    o = Origin()
    R = p.distance(o)
    R = R if R > 0 else 0.001 # Avoid singularities
    z = p.dz(o)
    P = 1.0

    # print(f"{R=}, {z=}")
    return 3*P*(z**3) / (2 * pi * (R**5))

p = Point_3d(1,1,1)
print(f"pressure = {bous(p)}")

pressure = 0.030629383078988458


In [3]:
def find_extreme(iterable, func=max) -> float|None:
    """
    Recursively find the min or max in nested iterables.
    :param iterable: The nested collection (list, tuple, etc.)
    :param func: The built-in min or max function
    """
    flat_elements = []
    
    for item in iterable:
        # Check if the item is a nested iterable (excluding strings if desired)
        if isinstance(item, (list, tuple, set)):
            # Recursive call to drill into the nested structure
            inner_extreme = find_extreme(item, func)
            if inner_extreme is not None:
                flat_elements.append(inner_extreme)
        else:
            # Base case: item is a single value (int, float, etc.)
            flat_elements.append(item)
            
    return func(flat_elements) if flat_elements else None

# Example usage:
nested_data = [1, [5, [10, -2]], 8, [0]]
print(f"Maximum: {find_extreme(nested_data, max)}") # Output: 10
print(f"Minimum: {find_extreme(nested_data, min)}") # Output: -2

Maximum: 10
Minimum: -2


In [4]:
X, Y, Z = np.mgrid[-2:2:40j, -2:2:40j, 1:5:40j]
x, y, z = X.flatten(), Y.flatten(), Z.flatten()

values = [bous(Point_3d(x=ix, y=iy, z=iz)) for ix, iy, iz in zip(x, y, z)]
# for value in values:
#     print(value)

isomin, isomax = find_extreme(values, min), find_extreme(values, max)
print(f"{isomin=}, {isomax=}")
fig = go.Figure(data=go.Volume(
    x=x,
    y=y,
    z=z,
    value=values,
    isomin=(isomin + (isomax - isomin)*0.05),
    isomax=isomax,
    opacity=0.2, # needs to be small to see through all surfaces
    surface_count=25, # needs to be a large number for good volume rendering,
    colorscale='Turbo'
    ))


fig.update_layout(title=f"Boussinesq Pressure",
                  width = 800, height=600,
                  scene=dict(
                      xaxis_title="x (in)",
                      yaxis_title = "y (in)",
                      zaxis_title = "z (in)",
                      zaxis=dict(autorange="reversed")
                    )
                  )
fig.show()

isomin=np.float64(0.001964875840640683), isomax=np.float64(0.4712438635658346)
